# Genomic Sequence to Expression Prediction

This notebook extracts sequence windows around transcription start sites, runs them through Enformer on the GPU, caches the embeddings, and trains a PyTorch MLP to predict tissue expression.

In [1]:
import os
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"

import gzip
import time
import numpy as np
import pandas as pd
import pysam
import tensorflow as tf
import tensorflow_hub as hub
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.notebook import tqdm

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

SEQ_LEN = 393_216

/home/igris/.local/lib/python3.12/site-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


### Load the Pretrained Enformer Model

In [2]:
enformer = hub.load(
    "https://www.kaggle.com/models/deepmind/enformer/TensorFlow2/enformer/1"
).model

### DNA One-Hot Encoder

In [3]:
def one_hot_encode(seq):
    arr = np.zeros((len(seq), 4), dtype=np.float32)
    seq_arr = np.frombuffer(seq.upper().encode('ascii'), dtype=np.uint8)
    arr[seq_arr == 65, 0] = 1.0
    arr[seq_arr == 67, 1] = 1.0
    arr[seq_arr == 71, 2] = 1.0
    arr[seq_arr == 84, 3] = 1.0
    return arr

### Sequence Extraction and Preprocessing (Parallelized)

In [4]:
def process_gene(args):
    gene_id, chrom, tss, gtex_df, tissue_cols = args
    f = pysam.FastaFile("GRCh38.primary_assembly.genome.fa")
    start = tss - 1 - SEQ_LEN // 2
    end = tss - 1 + SEQ_LEN // 2
    
    if start < 0 or end > f.get_reference_length(chrom):
        return None
        
    seq = f.fetch(chrom, start, end)
    if len(seq) != SEQ_LEN:
        return None
        
    x = one_hot_encode(seq)
    tpm = gtex_df.loc[gene_id, tissue_cols].values.astype(np.float32)
    y = np.log2(tpm + 1.0)
    return gene_id, chrom, tss, x, y

### Load Gene Annotations and Expression Matrix

In [5]:
gtex_file = "GTEx_Analysis_2025-08-22_v11_RNASeQCv2.4.3_gene_median_tpm.gct.gz"
gtex_df = pd.read_csv(gtex_file, sep='\t', skiprows=2)
gtex_gene_ids = set(gtex_df['Name'])
gtex_df = gtex_df.set_index('Name')
tissue_cols = gtex_df.columns[1:]

gtf_file = "gencode.v50.annotation.gtf.gz"
genes_meta = {}
with gzip.open(gtf_file, 'rt') as f:
    for line in f:
        if line.startswith('#'):
            continue
        parts = line.strip().split('\t')
        if len(parts) < 9 or parts[2] != 'gene':
            continue
        
        attrs = {}
        for item in parts[8].strip().split(';'):
            item = item.strip()
            if not item:
                continue
            try:
                k, v = item.split(' ', 1)
                attrs[k] = v.replace('"', '')
            except ValueError:
                pass
        
        gene_id = attrs.get('gene_id')
        if gene_id and gene_id in gtex_gene_ids:
            chrom = parts[0]
            start = int(parts[3])
            end = int(parts[4])
            strand = parts[6]
            tss = start if strand == '+' else end
            genes_meta[gene_id] = {
                'chrom': chrom,
                'tss': tss,
                'strand': strand,
                'gene_name': attrs.get('gene_name', ''),
                'gene_type': attrs.get('gene_type', '')
            }

### Extract Sequence Embeddings

In [6]:
df_rows = [{'gene_id': g_id, **info} for g_id, info in genes_meta.items() 
           if info['gene_type'] == 'protein_coding']
filtered_df = pd.DataFrame(df_rows).sort_values(by=['chrom', 'tss'])

embeddings_list = []
targets_list = []
processed_gene_ids = []
processed_metadata = []

print("Starting GPU inference pipeline with dynamic CPU prefetching...")
with ProcessPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(process_gene, (row['gene_id'], row['chrom'], row['tss'], gtex_df, tissue_cols)): row['gene_id'] 
               for idx, row in filtered_df.iterrows()}
    
    for future in tqdm(as_completed(futures), total=len(futures), desc="Extracting"):
        res = future.result()
        if res is None:
            continue
        gene_id, chrom, tss, x, y = res
        
        x_batch = np.expand_dims(x, axis=0)
        with tf.device('/GPU:0'):
            preds = enformer.predict_on_batch(x_batch)
            human_preds = preds['human'].numpy()
            
        emb = human_preds.mean(axis=1).squeeze()
        embeddings_list.append(emb)
        targets_list.append(y)
        processed_gene_ids.append(gene_id)
        processed_metadata.append({'gene_id': gene_id, 'chrom': chrom, 'tss': tss})

X = np.array(embeddings_list)
y = np.array(targets_list)
meta_df = pd.DataFrame(processed_metadata)

os.makedirs("data_embeddings", exist_ok=True)
np.save("data_embeddings/embeddings.npy", X)
np.save("data_embeddings/targets.npy", y)
meta_df.to_csv("data_embeddings/metadata.csv", index=False)

Starting GPU inference pipeline with dynamic CPU prefetching...


Extracting:   0%|          | 0/2669 [00:00<?, ?it/s]

### Train down-stream PyTorch prediction head with Early Stopping

In [7]:
class GenomicDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class ExpressionMLP(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, output_dim)
        )
    def forward(self, x):
        return self.network(x)

val_chroms = ["chr8", "chr9"]
test_chroms = ["chr21", "chr22"]
val_mask = meta_df["chrom"].isin(val_chroms).values
test_mask = meta_df["chrom"].isin(test_chroms).values
train_mask = ~(val_mask | test_mask)

if train_mask.sum() == 0:
    indices = np.random.permutation(len(X))
    split1, split2 = int(0.7 * len(X)), int(0.85 * len(X))
    X_train, y_train = X[indices[:split1]], y[indices[:split1]]
    X_val, y_val = X[indices[split1:split2]], y[indices[split1:split2]]
    X_test, y_test = X[indices[split2:]], y[indices[split2:]]
else:
    X_train, y_train = X[train_mask], y[train_mask]
    X_val, y_val = X[val_mask], y[val_mask]
    X_test, y_test = X[test_mask], y[test_mask]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_loader = DataLoader(GenomicDataset(X_train, y_train), batch_size=8, shuffle=True)
val_loader = DataLoader(GenomicDataset(X_val, y_val), batch_size=8, shuffle=False)

model = ExpressionMLP(X.shape[1], y.shape[1]).to(device)
criterion = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3)

best_loss = float("inf")
patience, patience_counter = 10, 0
best_state = None

for epoch in tqdm(range(100), desc="Training"):
    model.train()
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        loss = criterion(model(bx), by)
        loss.backward()
        optimizer.step()
        
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for bx, by in val_loader:
            bx, by = bx.to(device), by.to(device)
            val_loss += criterion(model(bx), by).item() * bx.size(0)
    val_loss /= len(val_loader.dataset)
    
    if val_loss < best_loss:
        best_loss = val_loss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

if best_state is not None:
    model.load_state_dict(best_state)
    torch.save(best_state, "model.pt")
    print("Saved best model weights to model.pt")

Training:   0%|          | 0/100 [00:00<?, ?it/s]

Early stopping at epoch 27
Saved best model weights to model.pt
